# Analyzing Startup Companies

In this lab, you will analyze a real-world dataset -- information about startup companies. The source of this dataset is [jSONAR](http://jsonstudio.com/resources/).

## Parsing the Data

As you can see, the schema is fairly complicated -- it has a bunch of fields, nested objects, arrays, and so on. It describes the company's products, key people, acquisition data, and more. We are going to use Spark SQL to infer the schema of this JSON document, and then issue queries using a natural SQL syntax.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('AnalyzingStartupApp').getOrCreate()
sc = spark.sparkContext


Create a `DataFrame` from the JSON file so that its schema is automatically inferred, print out the resulting schema, and register it as a temporary table called "companies".

In [2]:
from pyspark.sql import SQLContext
sqlContext = SQLContext(sc)

companies = sqlContext.read.json("data/companies.json")
companies.printSchema()
companies.registerTempTable("companies")

root
 |-- _id: struct (nullable = true)
 |    |-- $oid: string (nullable = true)
 |-- acquisition: struct (nullable = true)
 |    |-- acquired_day: long (nullable = true)
 |    |-- acquired_month: long (nullable = true)
 |    |-- acquired_year: long (nullable = true)
 |    |-- acquiring_company: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- permalink: string (nullable = true)
 |    |-- price_amount: long (nullable = true)
 |    |-- price_currency_code: string (nullable = true)
 |    |-- source_description: string (nullable = true)
 |    |-- source_url: string (nullable = true)
 |    |-- term_code: string (nullable = true)
 |-- acquisitions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- acquired_day: long (nullable = true)
 |    |    |-- acquired_month: long (nullable = true)
 |    |    |-- acquired_year: long (nullable = true)
 |    |    |-- company: struct (nullable = true)
 |    |    |    |-- name: strin

## Querying the Data

let's find out all startup companies info, such as name, number of employees, and homepage

In [3]:
df = sqlContext.sql("select name, number_of_employees, homepage_url from companies")
df.collect()

[Row(name='Wetpaint', number_of_employees=47, homepage_url='http://wetpaint-inc.com'),
 Row(name='AdventNet', number_of_employees=600, homepage_url='http://adventnet.com'),
 Row(name='Zoho', number_of_employees=1600, homepage_url='http://zoho.com'),
 Row(name='Digg', number_of_employees=60, homepage_url='http://www.digg.com'),
 Row(name='Facebook', number_of_employees=5299, homepage_url='http://facebook.com'),
 Row(name='Omnidrive', number_of_employees=None, homepage_url='http://www.omnidrive.com'),
 Row(name='Postini', number_of_employees=None, homepage_url='http://postini.com'),
 Row(name='Geni', number_of_employees=18, homepage_url='http://www.geni.com'),
 Row(name='Flektor', number_of_employees=None, homepage_url='http://www.flektor.com'),
 Row(name='Fox Interactive Media', number_of_employees=0, homepage_url='http://www.newscorp.com'),
 Row(name='Twitter', number_of_employees=1300, homepage_url='http://twitter.com'),
 Row(name='StumbleUpon', number_of_employees=None, homepage_url=

Let's talk about the money; figure out what the average acquisition price was.

In [7]:
sqlContext.sql("select avg(acquisition.price_amount) from companies").collect()

[Row(avg(acquisition.price_amount AS `price_amount`)=480597493.6224852)]

Let's get some additional detail -- print the average acquisition price grouped by number of years the company was active.

In [8]:
sqlContext.sql(
    """select   acquisition.acquired_year-founded_year as years_active,
                avg(acquisition.price_amount) as acq_price
       from     companies
       where    acquisition.price_amount is not null
       group by acquisition.acquired_year-founded_year
       order by acq_price desc""").collect()

[Row(years_active=52, acq_price=6500000000.0),
 Row(years_active=46, acq_price=4300000000.0),
 Row(years_active=33, acq_price=4200000000.0),
 Row(years_active=20, acq_price=3723475000.0),
 Row(years_active=162, acq_price=3600000000.0),
 Row(years_active=29, acq_price=2700000000.0),
 Row(years_active=26, acq_price=2657000000.0),
 Row(years_active=23, acq_price=2118000000.0),
 Row(years_active=34, acq_price=1957520000.0),
 Row(years_active=27, acq_price=1902520000.0),
 Row(years_active=21, acq_price=1821200000.0),
 Row(years_active=13, acq_price=1651454150.0),
 Row(years_active=41, acq_price=1300000000.0),
 Row(years_active=31, acq_price=1169500000.0),
 Row(years_active=25, acq_price=1065000000.0),
 Row(years_active=40, acq_price=720000000.0),
 Row(years_active=83, acq_price=575000000.0),
 Row(years_active=14, acq_price=566185000.0),
 Row(years_active=28, acq_price=548500000.0),
 Row(years_active=None, acq_price=541470444.8422939),
 Row(years_active=16, acq_price=494945000.0),
 Row(years

Finally, let's try to figure out the relationship between the company's total funding and acquisition price. In order to do that, you'll need a UDF (user-defined function) that, given a company, returns the sum of all its funding rounds. First, build that function and register it with the name "total_funding".

In [10]:
from pyspark.sql.types import IntegerType

sqlContext.registerFunction("total_funding", lambda investments: sum(
      [inv.funding_round.raised_amount or 0 for inv in investments]
    ), IntegerType())

<function __main__.<lambda>(investments)>

Test your function by retrieving the total funding for a few companies, such as Facebook, Paypal, and Alibaba. 

In [11]:
sqlContext.sql("""select total_funding(investments) from companies where 
    name = 'Facebook' """).collect()

[Row(total_funding(investments)=2050000)]

In [12]:
sqlContext.sql("""select total_funding(investments) from companies where 
    name = 'Alibaba' """).collect()

[Row(total_funding(investments)=711000000)]

In [13]:
sqlContext.sql("""select total_funding(investments) from companies where 
    name = 'PayPal' """).collect()

[Row(total_funding(investments)=31100000)]

Now, find the average ratio between the acquisition price and the total funding (which, in a simplistic way, represents return on investment).

In [30]:
ret = sqlContext.sql(
    """select avg(acquisition.price_amount/total_funding(investments))
       from   companies
       where  acquisition.price_amount is not null
       and    total_funding(investments) != 0""").collect()

In [31]:
print(ret)

[Row(avg((CAST(acquisition.price_amount AS `price_amount` AS DOUBLE) / CAST(total_funding(investments) AS DOUBLE)))=413.68840419548735)]
